# Final fraud detection submissions

This notebook reproduces the two selected Kaggle submissions: the chronological stack (`experiment18_stacking.csv`) and the equal-weight Spearman rank blend (`blend_exp12_18_25.csv`). The blend requires the original Experiment 12 HGB/CatBoost ensemble and Experiment 25 pseudo-labelled stack. 

Run from top to bottom after installing the repository dependencies and placing the competition CSVs in `data/raw/`. The output files are written to `final_outputs/`.

The notebook follows the original experiments' hyperparameters, chronological folds, and prediction logic. Exact byte-for-byte equality with the historical Kaggle uploads has not been verified; compare the new predictions against the original CSVs before treating them as identical.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata

# Works from the repository root, notebooks/, or final/.
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'data' / 'raw' / 'train.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Could not find data/raw/train.csv in the current directory or its parents')
DATA = ROOT / 'data' / 'raw'
FINAL_OUT = ROOT / 'final_outputs'
FINAL_OUT.mkdir(exist_ok=True, parents=True)
assert all((DATA / f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])
print('Repository:', ROOT)
print('Output:', FINAL_OUT)


Repository: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection
Output: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/final_outputs


## Original temporal HGB/CatBoost ensemble

This section uses the original chronological PR-AUC-based blend selection, including the five CatBoost seeds. It generates the Experiment 12 (notebook 04) input to the final rank blend.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier

ROOT = ROOT
DATA = ROOT / 'data' / 'raw'
OUT = FINAL_OUT
OUT.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
features = [c for c in train.columns if c.startswith('feat_')]
labelled = train.dropna(subset=['label']).copy()
assert 'time_step' in labelled.columns
assert labelled['time_step'].notna().all()
assert set(labelled['label'].unique()).issubset({0, 1})
assert all(c in test.columns for c in features)
print('Labelled:', labelled.shape, 'Test:', test.shape, 'Features:', len(features))
print('Time steps:', labelled.time_step.min(), labelled.time_step.max())

Labelled: (31235, 168) Test: (15329, 168) Features: 165
Time steps: 1 35


In [3]:
SEEDS = [11, 42, 73, 101, 202]
PERIODS = [(15, 19), (20, 24), (25, 29)]
CAT_PARAMS = dict(iterations=700, depth=6, learning_rate=0.045,
                  l2_leaf_reg=5, loss_function='Logloss', verbose=False,
                  thread_count=-1)

def fit_hgb(x, y):
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                      max_leaf_nodes=31, l2_regularization=1.0,
                                      random_state=42)
    m.fit(x, y)
    return m

def fit_cat(x, y, seed):
    m = CatBoostClassifier(**CAT_PARAMS, random_seed=seed)
    m.fit(x, y)
    return m

def score(y, p):
    return dict(pr_auc=average_precision_score(y, p),
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan)


In [4]:
predictions = {}
rows = []
for lo, hi in PERIODS:
    tr = labelled[labelled.time_step < lo]
    va = labelled[labelled.time_step.between(lo, hi)]
    if tr.empty or va.empty or tr.label.nunique() < 2 or va.label.nunique() < 2:
        print('Skipping unusable split:', (lo, hi), len(tr), len(va))
        continue
    xtr, ytr = tr[features], tr.label.astype(int)
    xva, yva = va[features], va.label.astype(int)
    baseline = fit_hgb(xtr, ytr).predict_proba(xva)[:, 1]
    cat_preds = np.column_stack([fit_cat(xtr, ytr, seed).predict_proba(xva)[:, 1]
                                 for seed in SEEDS])
    cat_mean = cat_preds.mean(axis=1)
    period = f'{lo}-{hi}'
    predictions[period] = dict(y=yva.to_numpy(), hgb=baseline,
                               cat_each=cat_preds, cat_mean=cat_mean,
                               n_train=len(tr), n_valid=len(va))
    for name, p in [('HGB baseline', baseline), ('CatBoost seed ensemble', cat_mean)]:
        rows.append(dict(period=period, model=name, n_train=len(tr),
                         n_valid=len(va), fraud_rate=yva.mean(), **score(yva, p)))
validation_results = pd.DataFrame(rows)
display(validation_results)


,period,model,n_train,n_valid,fraud_rate,pr_auc,roc_auc
0,15-19,HGB baseline,14896,3093,0.163595,0.987262,0.995038
1,15-19,CatBoost seed ensemble,14896,3093,0.163595,0.988448,0.994105
2,20-24,HGB baseline,17989,5617,0.126046,0.880880,0.952049
3,20-24,CatBoost seed ensemble,17989,5617,0.126046,0.871870,0.935112
4,25-29,HGB baseline,23606,2775,0.234955,0.991189,0.997044
5,25-29,CatBoost seed ensemble,23606,2775,0.234955,0.992611,0.997157


In [5]:
BLEND_WEIGHTS = [0.0, 0.25, 0.50, 0.75, 1.0]  # CatBoost weight
blend_rows = []
for period, d in predictions.items():
    for w in BLEND_WEIGHTS:
        p = w * d['cat_mean'] + (1 - w) * d['hgb']
        blend_rows.append(dict(period=period, cat_weight=w, n_valid=d['n_valid'],
                               **score(d['y'], p)))
blend_results = pd.DataFrame(blend_rows)
display(blend_results.pivot(index='cat_weight', columns='period', values='pr_auc'))
summary = (blend_results.groupby('cat_weight')
           .agg(mean_pr_auc=('pr_auc', 'mean'), min_pr_auc=('pr_auc', 'min'),
                mean_roc_auc=('roc_auc', 'mean')).reset_index())
display(summary.sort_values(['mean_pr_auc', 'min_pr_auc'], ascending=False))
# Predeclared selection rule: maximise mean period PR-AUC, break ties on worst period.
selected_weight = float(summary.sort_values(['mean_pr_auc', 'min_pr_auc'],
                                            ascending=False).iloc[0].cat_weight)
print('Selected CatBoost weight:', selected_weight)
print('Note: selection and reporting use the same validation periods; these are not unbiased final estimates.')
assert np.isclose(selected_weight, 0.25), ('Original blend weight changed:', selected_weight)


period,15-19,20-24,25-29
cat_weight,,,
0.00,0.987262,0.880880,0.991189
0.25,0.988270,0.878462,0.993123
0.50,0.988604,0.876565,0.993646
0.75,0.988692,0.874458,0.993411
1.00,0.988448,0.871870,0.992611


,cat_weight,mean_pr_auc,min_pr_auc,mean_roc_auc
1,0.25,0.953285,0.878462,0.979766
0,0.00,0.953110,0.880880,0.981377
2,0.50,0.952938,0.876565,0.978457
3,0.75,0.952187,0.874458,0.977091
4,1.00,0.950976,0.871870,0.975458


Selected CatBoost weight: 0.25
Note: selection and reporting use the same validation periods; these are not unbiased final estimates.


In [6]:
seed_rows = []
for period, d in predictions.items():
    for j, seed in enumerate(SEEDS):
        seed_rows.append(dict(period=period, seed=seed,
                              **score(d['y'], d['cat_each'][:, j])))
    seed_rows.append(dict(period=period, seed='ensemble',
                          **score(d['y'], d['cat_mean'])))
seed_results = pd.DataFrame(seed_rows)
display(seed_results.pivot(index='seed', columns='period', values='pr_auc'))

period,15-19,20-24,25-29
seed,,,
11,0.987590,0.867922,0.991886
42,0.989604,0.871037,0.991609
73,0.986955,0.872501,0.992962
101,0.988145,0.869497,0.992632
202,0.988416,0.873104,0.991804
ensemble,0.988448,0.871870,0.992611


In [7]:
X = labelled[features]
y = labelled.label.astype(int)
Xt = test[features]
hgb_final = fit_hgb(X, y)
hgb_test = hgb_final.predict_proba(Xt)[:, 1]
if selected_weight > 0:
    cat_test = np.column_stack([fit_cat(X, y, seed).predict_proba(Xt)[:, 1]
                                for seed in SEEDS]).mean(axis=1)
else:
    cat_test = np.zeros(len(test))
final_prob = (1 - selected_weight) * hgb_test + selected_weight * cat_test
assert sample.columns.tolist() == ['index', 'target']
assert len(sample) == len(final_prob)
assert np.isfinite(final_prob).all() and ((final_prob >= 0) & (final_prob <= 1)).all()
submission = sample.copy()
submission['target'] = final_prob
path = OUT / 'experiment12_temporal_ensemble.csv'
if path.exists():
    raise FileExistsError(f'Will not overwrite existing final output: {path}')
submission.to_csv(path, index=False)
print('Saved:', path, 'Shape:', submission.shape, 'CatBoost weight:', selected_weight)
display(submission.head())

Saved: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/final_outputs/experiment12_temporal_ensemble.csv Shape: (15329, 2) CatBoost weight: 0.25


,index,target
0,0,0.000454
1,1,0.000962
2,2,0.000883
3,3,0.000304
4,4,0.000272


## Chronological stacking

This section reproduces the original three-model stack, including its out-of-fold logistic regression meta-model. Its submission is one of the two selected final models.

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

ROOT = ROOT
DATA = ROOT/'data'/'raw'
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
features = [c for c in train if c.startswith('feat_')]
labelled = train.dropna(subset=['label']).copy()
assert labelled.label.isin([0,1]).all()
assert {'time_step','txId'}.issubset(train.columns)
assert all(c in test.columns for c in features)
print('Labelled:',len(labelled),'Test:',len(test),'Features:',len(features))
print('Labelled time:',labelled.time_step.min(),labelled.time_step.max())

Labelled: 31235 Test: 15329 Features: 165
Labelled time: 1 35


In [9]:
SEEDS = [11,42,73,101,202]
HGB_PARAMS = dict(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=42)
CAT_PARAMS = dict(iterations=700,depth=6,learning_rate=0.045,l2_leaf_reg=5,loss_function='Logloss',verbose=False,thread_count=-1)
# LightGBM configuration used in this stacking experiment.
LGB_PARAMS = dict(n_estimators=700,learning_rate=0.045,num_leaves=31,max_depth=-1,
                  reg_lambda=5.0,random_state=42,verbosity=-1,n_jobs=-1)
META_WINDOWS = [(10,14),(15,19),(20,24)]
EVAL_WINDOWS = [(25,29),(30,35)]

def fit_predict_models(tr,va):
    Xtr=tr[features]; ytr=tr.label.astype(int); Xva=va[features]
    if ytr.nunique()!=2: raise ValueError('Training slice has only one class')
    h=HistGradientBoostingClassifier(**HGB_PARAMS).fit(Xtr,ytr)
    ph=h.predict_proba(Xva)[:,1]
    pcs=[]
    for seed in SEEDS:
        c=CatBoostClassifier(**CAT_PARAMS,random_seed=seed)
        c.fit(Xtr,ytr)
        pcs.append(c.predict_proba(Xva)[:,1])
    pc=np.mean(pcs,axis=0)
    l=LGBMClassifier(**LGB_PARAMS).fit(Xtr,ytr)
    pl=l.predict_proba(Xva)[:,1]
    return np.column_stack([ph,pc,pl])

def make_window(lo,hi):
    tr=labelled[labelled.time_step<lo]
    va=labelled[labelled.time_step.between(lo,hi)]
    if tr.empty or va.empty: raise ValueError(f'Empty split: {lo}-{hi}')
    print(f'Window {lo}-{hi}: train {len(tr)}, validation {len(va)}, prevalence {va.label.mean():.3%}',flush=True)
    p=fit_predict_models(tr,va)
    return va.label.to_numpy(dtype=int),p

def score(y,p):
    return average_precision_score(y,p)

def reference(p):
    return .75*p[:,0]+.25*p[:,1]

In [10]:
meta_parts=[]; meta_labels=[]; meta_ids=[]
for lo,hi in META_WINDOWS:
    y,p=make_window(lo,hi)
    meta_parts.append(p); meta_labels.append(y); meta_ids.extend([f'{lo}-{hi}']*len(y))
    print('  Exp12:',round(score(y,reference(p)),6),'LightGBM:',round(score(y,p[:,2]),6))
P_meta=np.vstack(meta_parts); y_meta=np.concatenate(meta_labels)
print('OOF meta rows:',len(y_meta),'fraud prevalence:',round(y_meta.mean(),4))

Window 10-14: train 11496, validation 3400, prevalence 14.676%
  Exp12: 0.992753 LightGBM: 0.991833
Window 15-19: train 14896, validation 3093, prevalence 16.360%
  Exp12: 0.98827 LightGBM: 0.988677
Window 20-24: train 17989, validation 5617, prevalence 12.605%
  Exp12: 0.878462 LightGBM: 0.876511
OOF meta rows: 12110 fraud prevalence: 0.1415


In [11]:
stacker=LogisticRegression(C=0.1,max_iter=2000,solver='lbfgs')
stacker.fit(P_meta,y_meta)
print('Intercept:',stacker.intercept_)
print('Coefficients [HGB, CatBoost, LightGBM]:',stacker.coef_)
print('OOF in-sample stack PR-AUC (NOT valid evaluation):',round(score(y_meta,stacker.predict_proba(P_meta)[:,1]),6))

Intercept: [-3.82596914]
Coefficients [HGB, CatBoost, LightGBM]: [[2.41373771 3.41327452 2.71307266]]
OOF in-sample stack PR-AUC (NOT valid evaluation): 0.945938


In [12]:
rows=[]; heldout={}
for lo,hi in EVAL_WINDOWS:
    y,p=make_window(lo,hi)
    pr=reference(p)
    ps=stacker.predict_proba(p)[:,1]
    heldout[(lo,hi)]=(y,p,pr,ps)
    rows.append(dict(period=f'{lo}-{hi}',n=len(y),prevalence=y.mean(),
                     exp12_pr=score(y,pr),stack_pr=score(y,ps),
                     delta_pr=score(y,ps)-score(y,pr),
                     exp12_roc=roc_auc_score(y,pr),stack_roc=roc_auc_score(y,ps)))
results=pd.DataFrame(rows)
display(results.round(6))
print('Mean PR-AUC baseline:',round(results.exp12_pr.mean(),6))
print('Mean PR-AUC stack:',round(results.stack_pr.mean(),6))
print('Mean delta:',round(results.delta_pr.mean(),6))

Window 25-29: train 23606, validation 2775, prevalence 23.495%
Window 30-35: train 26381, validation 4854, prevalence 15.925%


,period,n,prevalence,exp12_pr,stack_pr,delta_pr,exp12_roc,stack_roc
0,25-29,2775,0.234955,0.993123,0.993799,0.000676,0.997625,0.997780
1,30-35,4854,0.159250,0.991554,0.992635,0.001081,0.998122,0.998414


Mean PR-AUC baseline: 0.992338
Mean PR-AUC stack: 0.993217
Mean delta: 0.000879


In [13]:
IMPROVES_BOTH = bool((results.delta_pr>0).all())
print('Stack improves both later windows:',IMPROVES_BOTH)
if not IMPROVES_BOTH:
    print('The stack did not improve both validation windows.')
else:
    print('The stack improved both validation windows.')

Stack improves both later windows: True
The stack improved both validation windows.


In [14]:
MAKE_SUBMISSION=True
if MAKE_SUBMISSION:
    if not IMPROVES_BOTH:
        raise RuntimeError('Stack did not improve both later windows; submission blocked.')
    # Test rows do not have labels; fit_predict_models needs only feature columns on va.
    P_test=fit_predict_models(labelled,test)
    predictions=stacker.predict_proba(P_test)[:,1]
    assert len(predictions)==len(test)==len(sample)
    assert list(sample.columns)==['index','target'], f'Unexpected schema: {list(sample.columns)}'
    submission=sample.copy()
    submission['target']=predictions
    assert submission.target.notna().all() and submission.target.between(0,1).all()
    output=FINAL_OUT/'experiment18_stacking.csv'
    output.parent.mkdir(parents=True,exist_ok=True)
    if output.exists(): raise FileExistsError(f'Refusing to overwrite {output}')
    submission.to_csv(output,index=False)
    print('Saved:',output)
else:
    print('Submission generation is disabled.')

Saved: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/final_outputs/experiment18_stacking.csv


## Full-stack pseudo-labelling

This section reproduces the original pseudo-labelled challenger used as the third input to the Spearman blend. It includes the original validation screening and cache logic.

In [15]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

ROOT = ROOT
DATA=ROOT/'data'/'raw'
train=pd.read_csv(DATA/'train.csv')
test=pd.read_csv(DATA/'test.csv')
sample=pd.read_csv(DATA/'sample_submission.csv')
labelled=train.dropna(subset=['label']).copy()
features=[c for c in train.columns if c.startswith('feat_')]
assert {'txId','time_step','label'}.issubset(train.columns)
assert labelled.txId.is_unique and labelled.label.isin([0,1]).all()
assert all(c in test for c in features)
assert list(sample.columns)==['index','target']
assert len(sample)==len(test)
print('Labelled:',len(labelled),'Test:',len(test),'Features:',len(features))

Labelled: 31235 Test: 15329 Features: 165


In [16]:
SEEDS=[11,42,73,101,202]
HGB_PARAMS=dict(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=42)
CAT_PARAMS=dict(iterations=700,depth=6,learning_rate=0.045,l2_leaf_reg=5,loss_function='Logloss',verbose=False,thread_count=-1)
LGB_PARAMS=dict(n_estimators=700,learning_rate=0.045,num_leaves=31,max_depth=-1,reg_lambda=5.0,random_state=42,verbosity=-1,n_jobs=-1)
OOF_WINDOWS=[(7,9),(10,12),(13,14),(15,17),(18,20),(21,21),(22,24),(25,29),(30,35)]
EVALUATIONS=[('long14',14,15,28),('long21',21,22,35),('recent25',24,25,29),('recent30',29,30,35)]
LOW,HIGH=0.02,0.98
PSEUDO_WEIGHT=0.1
STUDENT_BLEND=0.1
CLASS_CAP_FRACTION=0.10
CACHE=ROOT/'outputs'/'experiment25_cache'
CACHE.mkdir(parents=True,exist_ok=True)
config=repr((SEEDS,HGB_PARAMS,CAT_PARAMS,LGB_PARAMS,features))
data_hash=pd.util.hash_pandas_object(labelled[['txId','time_step','label']+features],index=True).values
fingerprint=hashlib.sha256(config.encode()+data_hash.tobytes()).hexdigest()[:16]
print('Cache fingerprint:',fingerprint)

def base_predictions(tr,va):
    Xtr=tr[features];ytr=tr.label.astype(int);Xva=va[features]
    if ytr.nunique()!=2:raise ValueError('Only one class in training window')
    ph=HistGradientBoostingClassifier(**HGB_PARAMS).fit(Xtr,ytr).predict_proba(Xva)[:,1]
    pc=np.mean([CatBoostClassifier(**CAT_PARAMS,random_seed=s).fit(Xtr,ytr).predict_proba(Xva)[:,1] for s in SEEDS],axis=0)
    pl=LGBMClassifier(**LGB_PARAMS).fit(Xtr,ytr).predict_proba(Xva)[:,1]
    return np.column_stack([ph,pc,pl])

def window_predictions(lo,hi,train_end):
    assert train_end<lo
    tr=labelled[labelled.time_step<=train_end]
    va=labelled[labelled.time_step.between(lo,hi)].copy()
    if tr.empty or va.empty:raise ValueError(f'Empty training/evaluation slice: {train_end} -> {lo}-{hi}')
    key=f'{fingerprint}_tr{train_end}_eval{lo}_{hi}.npz'
    path=CACHE/key
    ids=np.asarray(va.txId.astype(str).tolist(),dtype=str)
    y=va.label.to_numpy(dtype=int)
    if path.exists():
        try:
            with np.load(path,allow_pickle=False) as z:
                p=z['p'];cached_ids=z['ids'];cached_y=z['y']
            if not(np.array_equal(ids,cached_ids) and np.array_equal(y,cached_y) and p.shape==(len(y),3)):
                raise ValueError('Stale cache')
            print('Loaded:',key)
            return va,p
        except (ValueError,KeyError) as exc:
            print('Rebuilding cache:',type(exc).__name__)
    print(f'Training base models <= {train_end}; predicting {lo}-{hi}',flush=True)
    p=base_predictions(tr,va)
    np.savez_compressed(path,ids=ids,y=y,p=p)
    return va,p

def fit_meta(origin):
    parts=[];targets=[];used=[]
    for lo,hi in OOF_WINDOWS:
        if hi>origin:continue
        va,p=window_predictions(lo,hi,lo-1)
        parts.append(p);targets.append(va.label.to_numpy(dtype=int));used.append((lo,hi))
    if not parts:raise ValueError('No OOF windows')
    meta=LogisticRegression(C=0.1,max_iter=2000,solver='lbfgs').fit(np.vstack(parts),np.concatenate(targets))
    return meta,used,sum(len(y) for y in targets)

def select_pseudo(prob,n_train):
    cap=max(1,int(n_train*CLASS_CAP_FRACTION))
    neg=np.flatnonzero(prob<=LOW)
    pos=np.flatnonzero(prob>=HIGH)
    neg=neg[np.argsort(prob[neg],kind='stable')[:cap]]
    pos=pos[np.argsort(-prob[pos],kind='stable')[:cap]]
    ix=np.sort(np.concatenate([neg,pos]))
    return ix,(prob[ix]>=HIGH).astype(int),len(neg),len(pos)

def student_blend(historical,future,teacher):
    ix,yp,nn,np_=select_pseudo(teacher,len(historical))
    if len(ix)==0:
        return teacher.copy(),dict(selected=0,negative=0,positive=0)
    X=pd.concat([historical[features],future.iloc[ix][features]],ignore_index=True)
    y=np.concatenate([historical.label.to_numpy(dtype=int),yp])
    weights=np.concatenate([np.ones(len(historical)),np.full(len(ix),PSEUDO_WEIGHT)])
    student=HistGradientBoostingClassifier(**HGB_PARAMS).fit(X,y,sample_weight=weights)
    ps=student.predict_proba(future[features])[:,1]
    return (1-STUDENT_BLEND)*teacher+STUDENT_BLEND*ps,dict(selected=len(ix),negative=nn,positive=np_)

def metrics(y,p):
    if len(np.unique(y))<2:return np.nan,np.nan
    return average_precision_score(y,p),roc_auc_score(y,p)

Cache fingerprint: 4f838a69340df870


In [17]:
rows=[];checks=[];predictions={}
for name,origin,lo,hi in EVALUATIONS:
    print('\n===',name,'origin',origin,'eval',lo,hi,'===',flush=True)
    meta,windows,n_oof=fit_meta(origin)
    future,P=window_predictions(lo,hi,origin)
    teacher=meta.predict_proba(P)[:,1]
    historical=labelled[labelled.time_step<=origin]
    challenger,counts=student_blend(historical,future,teacher)
    predictions[name]=(future,teacher,challenger)
    checks.append(dict(window=name,origin=origin,meta_oof_rows=n_oof,meta_windows=str(windows),**counts))
    y=future.label.to_numpy(dtype=int)
    phases=[('full',np.ones(len(future),dtype=bool))]
    if name.startswith('long'):
        phases.extend([('early',future.time_step.between(origin+1,origin+7).to_numpy()),
                       ('late',future.time_step.between(origin+8,origin+14).to_numpy())])
    for phase,mask in phases:
        bpr,broc=metrics(y[mask],teacher[mask]);cpr,croc=metrics(y[mask],challenger[mask])
        rows.append(dict(window=name,phase=phase,n=int(mask.sum()),prevalence=float(y[mask].mean()),
                         baseline_pr=bpr,pseudo_pr=cpr,delta_pr=cpr-bpr,
                         baseline_roc=broc,pseudo_roc=croc,delta_roc=croc-broc))
results=pd.DataFrame(rows)
coverage=pd.DataFrame(checks)
display(coverage)
display(results.round(6))


=== long14 origin 14 eval 15 28 ===
Loaded: 4f838a69340df870_tr6_eval7_9.npz
Loaded: 4f838a69340df870_tr9_eval10_12.npz
Loaded: 4f838a69340df870_tr12_eval13_14.npz
Loaded: 4f838a69340df870_tr14_eval15_28.npz

=== long21 origin 21 eval 22 35 ===
Loaded: 4f838a69340df870_tr6_eval7_9.npz
Loaded: 4f838a69340df870_tr9_eval10_12.npz
Loaded: 4f838a69340df870_tr12_eval13_14.npz
Loaded: 4f838a69340df870_tr14_eval15_17.npz
Loaded: 4f838a69340df870_tr17_eval18_20.npz
Loaded: 4f838a69340df870_tr20_eval21_21.npz
Loaded: 4f838a69340df870_tr21_eval22_35.npz

=== recent25 origin 24 eval 25 29 ===
Loaded: 4f838a69340df870_tr6_eval7_9.npz
Loaded: 4f838a69340df870_tr9_eval10_12.npz
Loaded: 4f838a69340df870_tr12_eval13_14.npz
Loaded: 4f838a69340df870_tr14_eval15_17.npz
Loaded: 4f838a69340df870_tr17_eval18_20.npz
Loaded: 4f838a69340df870_tr20_eval21_21.npz
Loaded: 4f838a69340df870_tr21_eval22_24.npz
Loaded: 4f838a69340df870_tr24_eval25_29.npz

=== recent30 origin 29 eval 30 35 ===
Loaded: 4f838a69340df870

,window,origin,meta_oof_rows,meta_windows,selected,negative,positive
0,long14,14,6546,"[(7, 9), (10, 12), (13, 14)]",841,0,841
1,long21,21,11180,"[(7, 9), (10, 12), (13, 14), (15, 17), (18, 20...",1145,0,1145
2,recent25,24,15256,"[(7, 9), (10, 12), (13, 14), (15, 17), (18, 20...",552,0,552
3,recent30,29,18031,"[(7, 9), (10, 12), (13, 14), (15, 17), (18, 20...",613,0,613


,window,phase,n,prevalence,baseline_pr,pseudo_pr,delta_pr,baseline_roc,pseudo_roc,delta_roc
0,long14,full,10311,0.149064,0.921846,0.922394,0.000547,0.956369,0.958385,0.002016
1,long14,early,4634,0.186880,0.980313,0.980265,-0.000048,0.992303,0.992235,-0.000068
2,long14,late,5677,0.118196,0.836568,0.837635,0.001066,0.913673,0.918395,0.004722
3,long21,full,11705,0.151474,0.803385,0.809310,0.005925,0.882974,0.896909,0.013935
4,long21,early,5677,0.118196,0.877244,0.879765,0.002521,0.946175,0.951576,0.005401
5,long21,late,6028,0.182814,0.757596,0.766545,0.008949,0.821307,0.843874,0.022566
6,recent25,full,2775,0.234955,0.993671,0.993759,0.000088,0.997787,0.997630,-0.000157
7,recent30,full,4854,0.159250,0.992336,0.992309,-0.000027,0.998309,0.998240,-0.000068


In [18]:
full=results[results.phase=='full'].copy()
late=results[results.phase=='late'].copy()
long=full[full.window.str.startswith('long')]
recent=full[full.window.str.startswith('recent')]
print('Long-horizon full-window gains:')
display(long[['window','baseline_pr','pseudo_pr','delta_pr','delta_roc']].round(6))
print('Long-horizon late-half gains:')
display(late[['window','delta_pr','delta_roc']].round(6))
print('Recent-window gains (already reused in prior experiments):')
display(recent[['window','delta_pr','delta_roc']].round(6))
LONG_OK=(len(long)==2 and (long.delta_pr>0).all() and len(late)==2 and (late.delta_pr>=-0.0005).all())
RECENT_OK=(len(recent)==2 and (recent.delta_pr>=-0.0005).all())
print('Long-horizon screen passes:',LONG_OK)
print('Recent robustness screen passes:',RECENT_OK)
print('Candidate merits manual review:',bool(LONG_OK and RECENT_OK))
print('Caution: these validation windows are NOT independent of prior experiment choices.')

Long-horizon full-window gains:


,window,baseline_pr,pseudo_pr,delta_pr,delta_roc
0,long14,0.921846,0.922394,0.000547,0.002016
3,long21,0.803385,0.809310,0.005925,0.013935


Long-horizon late-half gains:


,window,delta_pr,delta_roc
2,long14,0.001066,0.004722
5,long21,0.008949,0.022566


Recent-window gains (already reused in prior experiments):


,window,delta_pr,delta_roc
6,recent25,0.000088,-0.000157
7,recent30,-0.000027,-0.000068


Long-horizon screen passes: True
Recent robustness screen passes: True
Candidate merits manual review: True
Caution: these validation windows are NOT independent of prior experiment choices.


In [19]:
audit = []

for name, (future, teacher, challenger) in predictions.items():
    ix, pseudo_y, n_neg, n_pos = select_pseudo(
        teacher,
        len(labelled[labelled.time_step <= dict(
            (n, origin) for n, origin, _, _ in EVALUATIONS
        )[name]])
    )

    actual_y = future.iloc[ix]["label"].to_numpy(dtype=int)

    audit.append({
        "window": name,
        "selected": len(ix),
        "positive_precision": (
            actual_y.mean() if len(ix) else np.nan
        ),
        "mean_teacher_probability": (
            teacher[ix].mean() if len(ix) else np.nan
        ),
        "overall_fraud_rate": future["label"].mean()
    })

display(pd.DataFrame(audit).round(5))

,window,selected,positive_precision,mean_teacher_probability,overall_fraud_rate
0,long14,841,0.99643,0.99260,0.14906
1,long21,1145,0.98690,0.99591,0.15147
2,recent25,552,0.99457,0.99240,0.23495
3,recent30,613,0.99511,0.99401,0.15925


In [20]:
MAKE_SUBMISSION=True
if MAKE_SUBMISSION:
    if not (LONG_OK and RECENT_OK):
        raise RuntimeError('Validation screening did not pass. No submission generated.')
    # Ensure the Kaggle test is chronologically later than all labelled training observations.
    if not (test.time_step.min()>labelled.time_step.max()):
        raise RuntimeError('Unexpected time ordering: verify test time_step range.')
    final_origin=int(labelled.time_step.max())
    final_meta,used,n_oof=fit_meta(final_origin)
    print('Final meta OOF rows:',n_oof,'windows:',used)
    Ptest=base_predictions(labelled,test)
    teacher=final_meta.predict_proba(Ptest)[:,1]
    challenger,counts=student_blend(labelled,test,teacher)
    print('Final pseudo-label selection:',counts)
    if counts['selected']==0:raise RuntimeError('No confident test pseudo-labels; refusing unchanged submission')
    if not np.isfinite(challenger).all() or not np.all((challenger>=0)&(challenger<=1)):
        raise RuntimeError('Invalid prediction values')
    submission=sample.copy()
    submission['target']=challenger
    output=FINAL_OUT/'experiment25_pseudo_stack.csv'
    output.parent.mkdir(parents=True,exist_ok=True)
    if output.exists():raise FileExistsError(f'Refusing to overwrite {output}')
    submission.to_csv(output,index=False)
    print('Saved:',output)
else:
    print('Submission disabled. Keep Experiment 18 (public score 0.94914).')

Loaded: 4f838a69340df870_tr6_eval7_9.npz
Loaded: 4f838a69340df870_tr9_eval10_12.npz
Loaded: 4f838a69340df870_tr12_eval13_14.npz
Loaded: 4f838a69340df870_tr14_eval15_17.npz
Loaded: 4f838a69340df870_tr17_eval18_20.npz
Loaded: 4f838a69340df870_tr20_eval21_21.npz
Loaded: 4f838a69340df870_tr21_eval22_24.npz
Loaded: 4f838a69340df870_tr24_eval25_29.npz
Loaded: 4f838a69340df870_tr29_eval30_35.npz
Final meta OOF rows: 22885 windows: [(7, 9), (10, 12), (13, 14), (15, 17), (18, 20), (21, 21), (22, 24), (25, 29), (30, 35)]
Final pseudo-label selection: {'selected': 3686, 'negative': 3123, 'positive': 563}
Saved: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/final_outputs/experiment25_pseudo_stack.csv


## Spearman rank blend

This is the original Notebook 11 approach: compute percentile ranks for the three prediction vectors and take their equal-weight mean. Unlike the earlier wildcard-based file discovery, the input filenames below are explicit, so the newer HGB-only CSV cannot be selected accidentally. The `index` column and row order must match across submissions.

In [21]:
INPUT_FILES = {
    'exp12': FINAL_OUT / 'experiment12_temporal_ensemble.csv',
    'exp18': FINAL_OUT / 'experiment18_stacking.csv',
    'exp25': FINAL_OUT / 'experiment25_pseudo_stack.csv',
}
subs = {name: pd.read_csv(path) for name, path in INPUT_FILES.items()}
base = subs['exp18']
assert list(base.columns) == ['index', 'target']
for name, sub in subs.items():
    assert list(sub.columns) == ['index', 'target'], f'Unexpected columns in {name}'
    assert len(sub) == len(base), f'Row count mismatch: {name}'
    assert sub['index'].equals(base['index']), f'Row alignment mismatch: {name}'
    assert np.isfinite(sub['target']).all(), f'Invalid predictions: {name}'
    assert sub['target'].between(0, 1).all(), f'Out-of-range predictions: {name}'

R = pd.DataFrame({name: rankdata(sub['target'].to_numpy()) / len(sub)
                  for name, sub in subs.items()})
print('Prediction rank correlations:')
display(R.corr(method='spearman').round(4))
blend = base.copy()
blend['target'] = R.mean(axis=1)
blend_path = FINAL_OUT / 'blend_exp12_18_25.csv'
if blend_path.exists():
    raise FileExistsError(f'Will not overwrite existing final output: {blend_path}')
blend.to_csv(blend_path, index=False)
print('Saved:', blend_path)


Prediction rank correlations:


,exp12,exp18,exp25
exp12,1.0000,0.9922,0.9943
exp18,0.9922,1.0000,0.9888
exp25,0.9943,0.9888,1.0000


Saved: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/final_outputs/blend_exp12_18_25.csv


## Final output checks

The selected submissions are the Experiment 18 stack and the Spearman blend. These checks verify their schema, lengths and prediction ranges. They do **not** establish equality with the previously uploaded Kaggle files or reproduce their leaderboard scores, which require Kaggle's hidden labels.

In [22]:
for filename in ['experiment18_stacking.csv', 'blend_exp12_18_25.csv']:
    path = FINAL_OUT / filename
    sub = pd.read_csv(path)
    sample = pd.read_csv(DATA / 'sample_submission.csv')
    assert list(sub.columns) == list(sample.columns) == ['index', 'target']
    assert len(sub) == len(sample)
    assert sub['index'].equals(sample['index'])
    assert sub['target'].notna().all() and sub['target'].between(0, 1).all()
    print(filename, 'rows:', len(sub), 'min:', round(sub.target.min(),6),
          'max:', round(sub.target.max(),6))
print('All output checks passed.')


experiment18_stacking.csv rows: 15329 min: 0.021334 max: 0.991096
blend_exp12_18_25.csv rows: 15329 min: 0.00013 max: 0.999913
All output checks passed.


In [24]:
import pandas as pd
import numpy as np
from pathlib import Path

# Locate the repository root
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "final":
    PROJECT_ROOT = PROJECT_ROOT.parent

original = PROJECT_ROOT / "submissions"
generated = PROJECT_ROOT / "final_outputs"

files = [
    "experiment18_stacking.csv",
    "blend_exp12_18_25.csv",
]

for filename in files:
    a = pd.read_csv(original / filename)
    b = pd.read_csv(generated / filename)

    same_index = a["index"].equals(b["index"])
    diff = np.abs(a["target"].to_numpy() - b["target"].to_numpy())

    print(f"\n{filename}")
    print("Same row order:", same_index)
    print("Maximum prediction difference:", diff.max())
    print(
        "Numerically identical:",
        same_index and np.allclose(
            a["target"], b["target"],
            rtol=0, atol=1e-12
        )
    )


experiment18_stacking.csv
Same row order: True
Maximum prediction difference: 0.0
Numerically identical: True

blend_exp12_18_25.csv
Same row order: True
Maximum prediction difference: 0.0
Numerically identical: True
